# CSV

Simple ETL of download and save


## Setup


In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas
import pandas as pd
import duckdb
from google.colab import userdata
import os

%alias head head


## Fetching a CSV file


In [ ]:
url = 'https://www2.census.gov/programs-surveys/popest/datasets/2020-2025/cities/totals/sub-est2025.csv'
url



'https://www2.census.gov/programs-surveys/popest/datasets/2020-2025/cities/totals/sub-est2025.csv'

### Extract, transform, load ( ETL ) in one step


In [ ]:
pandas.read_csv( url, dtype="str", encoding='latin-1' ).to_csv( "sub-est2025.csv", index=False )


In [ ]:
ls -l

total 7296
drwxr-xr-x 1 root root    4096 Nov 12 16:40 sample_data/
-rw-r--r-- 1 root root 7464565 Nov 16 05:06 sub-est2025.csv


In [ ]:
head ./sub-est2025.csv

SUMLEV,STATE,COUNTY,PLACE,COUSUB,CONCIT,PRIMGEO_FLAG,FUNCSTAT,NAME,STNAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,POPESTIMATE2024,POPESTIMATE2025
040,01,000,00000,00000,00000,0,A,Alabama,Alabama,5025437,5032962,5050058,5076868,5117850,5163055,5193088
162,01,000,00124,00000,00000,0,A,Abbeville city,Alabama,2355,2355,2340,2338,2351,2391,2378
162,01,000,00460,00000,00000,0,A,Adamsville city,Alabama,4406,4394,4329,4264,4219,4208,4178
162,01,000,00484,00000,00000,0,A,Addison town,Alabama,661,659,665,669,666,672,669
162,01,000,00676,00000,00000,0,A,Akron town,Alabama,229,228,228,226,231,234,236
162,01,000,00820,00000,00000,0,A,Alabaster city,Alabama,33341,33398,33732,33890,34177,34463,34748
162,01,000,00988,00000,00000,0,A,Albertville city,Alabama,22390,22407,22524,22788,23159,23522,23710
162,01,000,01132,00000,00000,0,A,Alexander City city,Alabama,14856,14833,14700,14615,14503,14498,14515
162,01,000,01228,00000,00000,0,A,Aliceville city,Alabama,2177

#### Summary

We downloaded a CSV file from a website and saved it as a file on the Colab VM.

We now have a local copy of the CSV file that we can work with.


## Transfer CSV to Hugging Face




While it's nice to have a local copy on the VM, the Colab VM is ephemeral.  That is, when we disconnect or stop the notebook, any files that we have created are deleted.  One solution is to upload files to a website such as Hugging Face.


### Authenticate to Hugging Face


In [ ]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]} "

'hf_zp ... QPb '

In [ ]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


- CSV to local parquet
  - compression
- CSV to local parquet with schema
  - data consistency
- local parquet to remote on Hugging Face
- querying remote parquet
  - no data base server


## DuckDB: query a CSV file



In [ ]:
duckdb.query(f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
  LIMIT 10
""").show() ;


## DuckDB: query a CSV file with a CTE


In [ ]:
duckdb.query(f"""
  WITH
  medicare as (
    SELECT *
    FROM read_csv(
      "{csv_file}",
      types={{'Telephone Number': 'VARCHAR'}},
      auto_detect=True
    )
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare_abq
  ORDER BY "City/Town", NPI
  LIMIT 10
""").show()


In [ ]:
query = f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
""".strip()
print(query)


In [ ]:
rel = duckdb.sql(query)


## Convert CSV to single parquet file


In [ ]:
parquet_file = csv_file.replace(".csv",".parquet")
parquet_file


In [ ]:
export_query = f'''
  COPY rel TO '{parquet_file}'
  (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
'''
print(export_query)


In [ ]:
duckdb.sql(export_query)


In [ ]:
! du -ms {parquet_file} {csv_file}


## Verify that DuckDB can query the local parquet file


In [ ]:
query = f"SELECT * FROM read_parquet('{parquet_file}')"
query


In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found: {len(rel):,}")


In [ ]:
rel.show()


## Convert CSV to hive-partitioned parquet


In [ ]:
parquet_path = "medicare_DAC_national"
parquet_path


In [ ]:
export_query = f'''
  COPY rel TO '{parquet_path}'
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);
'''
print(export_query)


In [ ]:
duckdb.sql(export_query)


In [ ]:
! tree {parquet_path}


In [ ]:
! du -ms {parquet_path} {parquet_file} {csv_file}


## Verify that DuckDB can query hive parquet files


In [ ]:
dataset_path = f"{parquet_path}/**/*.parquet"
dataset_path


In [ ]:
query = f"SELECT * FROM read_parquet('{dataset_path}')"
query


In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found across all partitions: {len(rel):,}")


In [ ]:
rel.show()


## Push to Hugging Face


In [ ]:
stop


### Push CSV file


In [ ]:
os.environ["hf_csv"] = csv_file
! echo ${hf_csv}


In [ ]:
%%capture hf_upload_csv
%%bash

echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ${hf_csv} \
  ${hf_csv}


In [ ]:
print(hf_upload_csv.stdout)


### Push Parquet File

In [ ]:
os.environ["hf_parquet"] = parquet_file
! echo ${hf_parquet}


In [ ]:
%%capture hf_upload_parquet
%%bash

echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ./${hf_parquet} \
  ${hf_parquet}


In [ ]:
print(hf_upload_parquet.stdout)


### Push Hive Parquet Files

In [ ]:
os.environ["hf_hive"] = parquet_path
! echo ${hf_hive}


In [ ]:
%%capture hf_upload_hive
%%bash

echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ./${hf_hive} \
  ${hf_hive}


In [ ]:
print(hf_upload_hive.stdout)


## Query Hugging Face using DuckDB


### CSV File


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_csv = f"hf://datasets/rwcitek/medicare/{csv_file}"
hf_url_csv


In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(1)
  FROM read_csv('{hf_url_csv}')
"""
print(query)


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


### Parquet File


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_parquet = f"hf://datasets/rwcitek/medicare/{parquet_file}"
hf_url_parquet


In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(1)
  FROM read_parquet('{hf_url_parquet}')
"""
print(query)


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


In [ ]:
# Query the remote dataset directly
query = f"""
  DESCRIBE
  SELECT *
  FROM read_parquet('{hf_url_parquet}')
"""
print(query)


In [ ]:
duckdb.sql(query).show(max_rows=100)


### Hive Parquet Files


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_hive = f"hf://datasets/rwcitek/medicare/{parquet_path}/**/*.parquet"
hf_url_hive


In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(*)
  FROM read_parquet('{hf_url_hive}')
"""


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


In [ ]:
# Query the remote dataset directly
query = f"""
  DESCRIBE
  SELECT *
  FROM read_parquet('{hf_url_hive}')
"""
print(query)


In [ ]:
duckdb.sql(query).show(max_rows=100)
